# Soal 4 · WORKER: Spark Structured Streaming pada satu partisi Kafka
**Big Data** · 4 notebook Worker (NOMOR_WORKER 1–4) + 1 Master. Pola sama dengan Soal 3 dan materi *Kafka_Spark_Cluster_Perf*:
Worker **menunggu perintah Master** lewat topic kontrol (`MULAI` / `BERHENTI`), menjalankan **query streaming** atas
**partisi Kafka miliknya** (Worker n → partisi n−1 topic `wiki.recentchange.raw`), lalu mengirim hasil dan metrik ke Master.

| Bagian | Isi |
|---|---|
| 4a | `readStream` Kafka (`assign`) → `from_json` skema eksplisit → event time `ts` dari field JSON `timestamp` → watermark 2 menit → jendela 1 menit per wiki: jumlah edit dan edit bot → `foreachBatch` (append): simpan CSV per batch + kirim ke Master; query `memory` (update) untuk tampilan langsung; `checkpointLocation` |
| 4b | setiap progress (`recentProgress`) dikirim ke Master: input/processed rows per detik, durasi batch, watermark |
| 4c | checkpoint yang sama dipakai di setiap `MULAI`; opsi **crash buatan** setelah hasil terkirim tetapi sebelum commit |

**Syarat:** konektor `spark-sql-kafka-0-10_2.13:4.1.3` (sudah ada di cache PC Master), producer menyala.
**Isi Sel 1:** `NOMOR_WORKER`. **Urutan:** Sel 1 → 5 (Sel 5 terus berjalan sampai di-Interrupt).

## Sel 1. Konfigurasi, Kafka, dan pelapor

In [1]:
# ===== SEL 1: Konfigurasi, Kafka, dan pelapor (Worker) =====
import os, sys, json, time, socket, shutil, uuid
from pathlib import Path
from confluent_kafka import Producer, Consumer, KafkaError

# ---------------- ISI SESUAI WORKER ----------------
KELOMPOK     = "wiki4"
NOMOR_WORKER = 1                                   # 1 ... 4  -> partisi Kafka 0 ... 3
MASTER_IP    = "192.168.1.15"
# ---------------------------------------------------

BOOTSTRAP     = f"{MASTER_IP}:9092"
TOPIC_RAW     = "wiki.recentchange.raw"
TOPIC_KONTROL = f"{KELOMPOK}-kontrol"               # Master -> Worker: MULAI / BERHENTI
TOPIC_HASIL   = f"{KELOMPOK}-hasil"                 # Worker -> Master: jendela, progress, checkpoint
TOPIC_STATUS  = f"{KELOMPOK}-status"                # Worker -> Master: SIAP / HIDUP / QUERY_*
PARTISI       = NOMOR_WORKER - 1
HOST          = f"{socket.gethostname()}-w{NOMOR_WORKER}"
DASAR         = Path("C:/BigData/soal4") / f"w{NOMOR_WORKER}"
CKPT          = DASAR / "checkpoint"                # checkpointLocation query utama
HASIL_DIR     = DASAR / "hasil_batch"               # CSV hasil tiap batch (foreachBatch)
DASAR.mkdir(parents=True, exist_ok=True)

produser = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True, "linger.ms": 20})
galat_kirim = []
def _hasil_kirim(err, msg):
    if err is not None:
        galat_kirim.append(str(err))

def kirim(topic, key, obj):
    produser.produce(topic, key=key, value=json.dumps(obj, default=str).encode("utf-8"), on_delivery=_hasil_kirim)
    produser.poll(0)

KEADAAN = {"run": "-", "fase": "-"}
def lapor(status, **tambahan):
    kirim(TOPIC_STATUS, HOST, {"run": KEADAAN["run"], "worker": HOST, "nomor": NOMOR_WORKER, "partisi": PARTISI,
                               "status": status, "fase": KEADAAN["fase"], "waktu": time.time(), **tambahan})

meta = produser.list_topics(timeout=5)
print("Terhubung ke Kafka | topic kelompok:", sorted(t for t in meta.topics if t.startswith(KELOMPOK)))
print(f"Worker {HOST} -> partisi {PARTISI} dari {TOPIC_RAW} "
      f"({len(meta.topics[TOPIC_RAW].partitions) if TOPIC_RAW in meta.topics else '?'} partisi)")
print("Checkpoint:", CKPT.resolve())

Terhubung ke Kafka | topic kelompok: ['wiki4-hasil', 'wiki4-kontrol', 'wiki4-status']
Worker NthaanSnn-w1 -> partisi 0 dari wiki.recentchange.raw (4 partisi)
Checkpoint: C:\BigData\soal4\w1\checkpoint


## Sel 2. SparkSession dengan konektor Kafka, skema, dan query (Soal 4a)
`spark.sql.shuffle.partitions = 4` **tidak boleh diubah** setelah checkpoint dibuat (jumlah state store tersimpan di checkpoint).
Kolom `timestamp` bawaan Kafka (waktu pesan masuk broker) diganti nama menjadi `kafka_ts`; **event time** = field JSON `timestamp`.

In [2]:
# ===== SEL 2: SparkSession, skema, dan pembangun query =====
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, IntegerType, BooleanType, DoubleType)

spark = (SparkSession.builder.appName(f"{KELOMPOK}-worker{NOMOR_WORKER}").master("local[2]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3")
         .config("spark.driver.memory", "1g")
         .config("spark.driver.host", "127.0.0.1").config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "Asia/Jakarta")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark", spark.version, "| UI:", sc.uiWebUrl)

SKEMA = StructType([
    StructField("id", LongType()), StructField("type", StringType()), StructField("wiki", StringType()),
    StructField("server_name", StringType()), StructField("title", StringType()), StructField("namespace", IntegerType()),
    StructField("user", StringType()), StructField("is_anon", BooleanType()), StructField("anon_kind", StringType()),
    StructField("bot", BooleanType()), StructField("minor", BooleanType()),
    StructField("timestamp", LongType()),                    # EVENT TIME (detik epoch dari Wikimedia)
    StructField("length_old", LongType()), StructField("length_new", LongType()),
    StructField("comment", StringType()), StructField("ingest_ts", DoubleType()),
])
WATERMARK, JENDELA = "2 minutes", "1 minute"

def bangun_agregat():
    """Soal 4a: stream Kafka partisi milik Worker -> from_json -> event time -> watermark -> jendela 1 menit per wiki."""
    mentah = (spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers", BOOTSTRAP)
              .option("assign", json.dumps({TOPIC_RAW: [PARTISI]}))
              .option("startingOffsets", "latest")          # hanya berlaku saat checkpoint masih kosong
              .load())
    edit = (mentah.select(F.col("partition"), F.col("offset"), F.col("timestamp").alias("kafka_ts"),
                          F.from_json(F.col("value").cast("string"), SKEMA).alias("d"))
                  .select("partition", "offset", "kafka_ts", "d.*")
                  .filter("type = 'edit' AND wiki IS NOT NULL AND timestamp IS NOT NULL")
                  .withColumn("ts", F.timestamp_seconds("timestamp")))
    return (edit.withWatermark("ts", WATERMARK)
                .groupBy(F.window("ts", JENDELA).alias("w"), "wiki")
                .agg(F.count("*").alias("n_edit"),
                     F.sum(F.coalesce(F.col("bot").cast("int"), F.lit(0))).alias("n_bot")))

def ke_dict(p):
    """Progress Spark 4 bisa berupa objek; ubah ke dict biasa."""
    if isinstance(p, dict):
        return p
    j = getattr(p, "json", None)
    return json.loads(j() if callable(j) else j) if j is not None else json.loads(str(p))

print("Skema dan pembangun query siap. Jendela", JENDELA, "| watermark", WATERMARK)

Spark 4.1.3 | UI: http://127.0.0.1:4041
Skema dan pembangun query siap. Jendela 1 minute | watermark 2 minutes


## Sel 3. `foreachBatch`: simpan CSV per batch dan kirim ke Master
Mode **append**: satu jendela dikeluarkan **sekali**, setelah watermark melewati akhir jendela. CSV ditulis ke folder `batch_id=N`
dengan mode overwrite, sehingga batch yang diulang setelah restart **menimpa**, bukan menambah. Pesan ke Master berkunci
`(worker, akhir jendela, wiki)`; Master menimpa kiriman ganda dan menghitung jumlahnya.

**Crash buatan (4c):** bila diaktifkan oleh Master, batch tertentu error **setelah** hasil terkirim tetapi **sebelum** Spark
mencatat commit, sehingga batch itu diulang saat query dimulai lagi.

In [3]:
# ===== SEL 3: foreachBatch =====
CRASH = {"aktif": False, "setelah": 3, "batch_awal": None}
STAT = {"batch": -1, "jendela_terkirim": 0}

def proses_batch(df, batch_id):
    if CRASH["aktif"] and CRASH["batch_awal"] is None:
        CRASH["batch_awal"] = batch_id
    df.persist()
    hasil = df.select(F.date_format("w.start", "yyyy-MM-dd HH:mm").alias("mulai"),
                      F.unix_timestamp("w.end").alias("akhir"), "wiki", "n_edit", "n_bot")
    baris = [r.asDict() for r in hasil.collect()]
    if baris:
        (hasil.coalesce(1).write.mode("overwrite").option("header", True)
              .csv((HASIL_DIR / f"batch_id={batch_id}").resolve().as_uri()))
        galat_kirim.clear()
        for b in baris:
            kirim(TOPIC_HASIL, f"{NOMOR_WORKER}|{b['akhir']}|{b['wiki']}",
                  {"jenis": "jendela", "run": KEADAAN["run"], "worker": HOST, "nomor": NOMOR_WORKER, "partisi": PARTISI,
                   "fase": KEADAAN["fase"], "batch": batch_id, **b, "t_kirim": time.time()})
        produser.flush(30)
        STAT["jendela_terkirim"] += len(baris)
        teratas = sorted(baris, key=lambda b: -b["n_edit"])[:3]
        print(f"[batch {batch_id}] {len(baris)} jendela final terkirim | teratas: "
              + ", ".join(f"{b['mulai'][-5:]} {b['wiki']} {b['n_edit']}" for b in teratas))
    df.unpersist()
    STAT["batch"] = batch_id
    if CRASH["aktif"] and baris and batch_id >= CRASH["batch_awal"] + CRASH["setelah"]:
        CRASH["aktif"] = False
        print(f"!!! SIMULASI CRASH pada batch {batch_id}: hasil SUDAH terkirim, commit BELUM dicatat")
        raise RuntimeError(f"SIMULASI CRASH batch {batch_id}")

print("foreachBatch siap.")

foreachBatch siap.


## Sel 4. Mulai / hentikan query dan baca isi checkpoint

In [4]:
# ===== SEL 4: Kendali query =====
Q = {"utama": None, "memory": None, "terkirim": set(), "dilaporkan_mati": False}

def _json_terakhir(path):
    """Baris JSON terakhir di file checkpoint (format: 'v1' lalu JSON)."""
    try:
        baris = [b for b in Path(path).read_text(encoding="utf-8").splitlines() if b.strip().startswith("{")]
        return json.loads(baris[-1]) if baris else None
    except Exception:
        return None

def info_checkpoint():
    def nomor(d):
        return sorted(int(f.name) for f in (CKPT / d).glob("*") if f.name.isdigit()) if (CKPT / d).exists() else []
    off, com = nomor("offsets"), nomor("commits")
    meta_akhir = None
    if off:
        isi = [b for b in (CKPT / "offsets" / str(off[-1])).read_text(encoding="utf-8").splitlines() if b.startswith("{")]
        meta_akhir = json.loads(isi[0]) if isi else None
    return {"offsets": off[-6:], "commits": com[-6:], "n_offsets": len(off), "n_commits": len(com),
            "offset_awal": _json_terakhir(CKPT / "sources" / "0" / "0"),
            "offset_commit_akhir": _json_terakhir(CKPT / "offsets" / str(com[-1])) if com else None,
            "offset_batch_akhir": _json_terakhir(CKPT / "offsets" / str(off[-1])) if off else None,
            "metadata_batch_akhir": meta_akhir}

def kirim_progress():
    q = Q["utama"]
    if q is None:
        return
    for p in q.recentProgress:
        d = ke_dict(p)
        kunci = (d.get("runId"), d.get("batchId"), d.get("timestamp"))
        if kunci in Q["terkirim"]:
            continue
        Q["terkirim"].add(kunci)
        kirim(TOPIC_HASIL, f"{NOMOR_WORKER}|progress|{kunci}",
              {"jenis": "progress", "run": KEADAAN["run"], "worker": HOST, "nomor": NOMOR_WORKER, "partisi": PARTISI,
               "fase": KEADAAN["fase"], "progress": d})

def mulai(perintah):
    if Q["utama"] is not None and Q["utama"].isActive:
        print("Query sudah berjalan; perintah MULAI diabaikan."); return
    KEADAAN.update(run=perintah["run"], fase=perintah["fase"])
    if perintah.get("reset"):
        shutil.rmtree(CKPT, ignore_errors=True); shutil.rmtree(HASIL_DIR, ignore_errors=True)
        print("Checkpoint dan hasil lama dihapus (sesi baru).")
    CRASH.update(aktif=perintah.get("crash", False), setelah=perintah.get("crash_setelah", 3), batch_awal=None)
    trig = f"{perintah['trigger']} seconds"
    Q["utama"] = (bangun_agregat().writeStream.outputMode("append")
                  .foreachBatch(proses_batch)
                  .option("checkpointLocation", CKPT.resolve().as_uri())
                  .trigger(processingTime=trig).queryName(f"wiki4_utama_w{NOMOR_WORKER}").start())
    Q["memory"] = (bangun_agregat().writeStream.outputMode("update").format("memory")
                   .queryName(f"langsung_w{NOMOR_WORKER}").trigger(processingTime=trig).start())
    Q["dilaporkan_mati"] = False
    lapor("QUERY_MULAI", trigger=perintah["trigger"], crash=CRASH["aktif"], ckpt=info_checkpoint())
    print(f"MULAI fase {KEADAAN['fase']} | trigger {trig} | crash buatan: {CRASH['aktif']} | runId {Q['utama'].runId}")

def berhenti(alasan="perintah Master"):
    for nama in ("utama", "memory"):
        q = Q[nama]
        if q is not None and q.isActive:
            q.stop()
    kirim_progress()
    info = info_checkpoint()
    kirim(TOPIC_HASIL, f"{NOMOR_WORKER}|ckpt|{time.time()}",
          {"jenis": "checkpoint", "run": KEADAAN["run"], "worker": HOST, "nomor": NOMOR_WORKER, "partisi": PARTISI,
           "fase": KEADAAN["fase"], "alasan": alasan, "info": info, "waktu": time.time()})
    lapor("QUERY_BERHENTI", alasan=alasan)
    produser.flush(10)
    print(f"BERHENTI ({alasan}) | offsets {info['offsets']} | commits {info['commits']}")

print("Kendali query siap.")

Kendali query siap.


## Sel 5. Loop Worker: perintah Master + pelaporan progress
Sel ini **terus berjalan**. Hentikan (Interrupt) hanya setelah Master selesai.

In [5]:
# ===== SEL 5: Loop Worker =====
ctl = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"{KELOMPOK}-ctl-{HOST}-{uuid.uuid4().hex[:6]}",
                "auto.offset.reset": "latest", "enable.auto.commit": False})
ctl.subscribe([TOPIC_KONTROL], on_assign=lambda c, p: lapor("SIAP"))
print(f"Worker {HOST} (partisi {PARTISI}) menunggu perintah di '{TOPIC_KONTROL}' ... (Interrupt untuk berhenti)")

terakhir_hidup = terakhir_cetak = 0
try:
    while True:
        m = ctl.poll(0.5)
        if m is not None and not m.error():
            try:
                x = json.loads(m.value())
            except ValueError:
                x = {}
            if x.get("untuk", "semua") == "semua" or NOMOR_WORKER in x.get("untuk", []):
                if x.get("aksi") == "MULAI":
                    mulai(x)
                elif x.get("aksi") == "BERHENTI":
                    berhenti()
        q = Q["utama"]
        if q is not None:
            kirim_progress()
            if not q.isActive and not Q["dilaporkan_mati"]:
                Q["dilaporkan_mati"] = True
                galat = str(q.exception())[:300] if q.exception() else "berhenti"
                if q.exception():
                    print("Query utama MATI:", galat.splitlines()[0][:150])
                    berhenti(alasan="query mati: " + galat.splitlines()[0][:120])
        if time.time() - terakhir_hidup > 10:
            p = ke_dict(q.lastProgress) if q is not None and q.lastProgress else {}
            lapor("HIDUP", aktif=bool(q is not None and q.isActive), batch=p.get("batchId"),
                  input_rps=p.get("inputRowsPerSecond"), proses_rps=p.get("processedRowsPerSecond"),
                  watermark=(p.get("eventTime") or {}).get("watermark"))
            terakhir_hidup = time.time()
        if q is not None and q.isActive and time.time() - terakhir_cetak > 60:
            terakhir_cetak = time.time()
            try:                                   # tampilan langsung dari memory sink (mode update)
                spark.sql(f"SELECT date_format(w.start, 'HH:mm') AS mulai, wiki, n_edit, n_bot FROM langsung_w{NOMOR_WORKER} "
                          f"ORDER BY w.start DESC, n_edit DESC LIMIT 5").show(truncate=False)
            except Exception:
                pass
except KeyboardInterrupt:
    print("\nWorker dihentikan.")
finally:
    try:
        if Q["utama"] is not None and Q["utama"].isActive:
            berhenti(alasan="Worker dihentikan")
        lapor("BERHENTI"); produser.flush(10)
    finally:
        ctl.close()

Worker NthaanSnn-w1 (partisi 0) menunggu perintah di 'wiki4-kontrol' ... (Interrupt untuk berhenti)
Checkpoint dan hasil lama dihapus (sesi baru).
MULAI fase A-10s | trigger 10 seconds | crash buatan: False | runId d67abd26-3307-4783-9bcc-fe325c6c354c
+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----+------+-----+
+-----+----+------+-----+

+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----+------+-----+
+-----+----+------+-----+

+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----+------+-----+
+-----+----+------+-----+

+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----+------+-----+
+-----+----+------+-----+

+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----+------+-----+
+-----+----+------+-----+

BERHENTI (perintah Master) | offsets [0] | commits [0]
MULAI fase B-30s | trigger 30 seconds | crash buatan: False | runId febaefad-cd68-48e2-8ce1-4086f1ef3ad5
+-----+----+------+-----+
|mulai|wiki|n_edit|n_bot|
+-----+----